# Building a Mini-GPT — Pure Python
#
# This file builds a complete character-level GPT from scratch
# using only Python lists, loops, and the math module.
#
# A GPT is a DECODER-ONLY Transformer:
#   - It reads a sequence of token IDs
#   - Predicts the NEXT token at every position
#   - Uses CAUSAL MASKING so each position only sees prior tokens
#
# We will:
#   1. Build all the GPT components with loops
#   2. Run a forward pass on a real sentence
#   3. Implement a text generation loop (autoregressive inference)
#   4. Show a manual "training step" with finite-difference gradients
#      to illustrate how the loss actually drives learning

import math
import random

# ─────────────────────────────────────────────────────────────
# All utility functions from notebooks 01 and 02
# (copy-paste these or run those notebook cells first)
# ─────────────────────────────────────────────────────────────

def mat_mul(A, B):
    m, k, n = len(A), len(A[0]), len(B[0])
    C = [[0.0]*n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            C[i][j] = sum(A[i][c]*B[c][j] for c in range(k))
    return C

def transpose(M):
    return [[M[r][c] for r in range(len(M))] for c in range(len(M[0]))]

def softmax(scores):
    m = max(s for s in scores if s != float('-inf'))
    exps = [math.exp(s - m) if s != float('-inf') else 0.0 for s in scores]
    t = sum(exps)
    return [e/t if t > 0 else 0.0 for e in exps]

def make_zeros(rows, cols):
    return [[0.0]*cols for _ in range(rows)]

def add_vectors(a, b):
    return [x + y for x, y in zip(a, b)]

def make_random_matrix(rows, cols, scale=0.1):
    return [[random.gauss(0, scale) for _ in range(cols)] for _ in range(rows)]

def gelu(x):
    return 0.5 * x * (1.0 + math.tanh(math.sqrt(2.0/math.pi) * (x + 0.044715*x**3)))

class Linear:
    def __init__(self, in_f, out_f):
        scale = math.sqrt(2.0 / (in_f + out_f))
        self.W = make_random_matrix(out_f, in_f, scale=scale)
        self.b = [0.0] * out_f

    def __call__(self, x):
        out_f, in_f = len(self.W), len(self.W[0])
        return [
            [self.b[i] + sum(self.W[i][j]*token[j] for j in range(in_f))
             for i in range(out_f)]
            for token in x
        ]


class LayerNorm:
    def __init__(self, d, eps=1e-6):
        self.eps   = eps
        self.gamma = [1.0]*d
        self.beta  = [0.0]*d

    def __call__(self, x):
        out = []
        for token in x:
            n    = len(token)
            mean = sum(token)/n
            var  = sum((v-mean)**2 for v in token)/n
            std  = math.sqrt(var + self.eps)
            out.append([self.gamma[d]*(token[d]-mean)/std + self.beta[d]
                        for d in range(n)])
        return out


def make_causal_mask(seq_len):
    """Lower-triangular: position i can only attend to positions 0..i"""
    return [[1 if j <= i else 0 for j in range(seq_len)] for i in range(seq_len)]

# ─────────────────────────────────────────────────────────────
# SECTION 1: Token and Positional Embeddings
#
# The Transformer works on continuous vectors, not discrete token IDs.
# We map each integer token ID to a d_model-dimensional vector using
# an "embedding table" — just a regular list-of-lists.
#
# Token embedding:    lookup table of shape [vocab_size, d_model]
# Position embedding: lookup table of shape [max_seq_len, d_model]
#
# The final input to the Transformer is:
#   input[t] = token_embedding[token_id[t]] + position_embedding[t]
# ─────────────────────────────────────────────────────────────

class Embedding:
    """
    A lookup table mapping integer indices to float vectors.
    Think of it as a list of d_model-dimensional vectors, one per index.
    
    In PyTorch: nn.Embedding(num_embeddings, embedding_dim)
    """
    def __init__(self, num_embeddings, embedding_dim):
        # Each embedding vector is randomly initialized
        self.table = make_random_matrix(num_embeddings, embedding_dim, scale=0.02)
        self.embedding_dim = embedding_dim

    def __call__(self, indices):
        """
        indices: a list of integer token IDs, e.g. [3, 7, 2, 5]
        Returns: a list of embedding vectors [[...], [...], [...], [...]]
        """
        return [self.table[i][:] for i in indices]  # Copy each row


# Test Embedding
vocab_size  = 10
d_model     = 6
max_seq_len = 20

token_emb = Embedding(vocab_size, d_model)
pos_emb   = Embedding(max_seq_len, d_model)

token_ids = [3, 7, 2]  # A 3-token sequence

tok_vecs = token_emb(token_ids)               # 3 token embeddings
pos_vecs = pos_emb(list(range(len(token_ids))))  # Position 0, 1, 2

# Add token + position embeddings element-wise
x = [add_vectors(tok_vecs[t], pos_vecs[t]) for t in range(len(token_ids))]

print("=" * 55)
print("DEMO: Token + Positional Embeddings")
print("=" * 55)
print(f"Token IDs:          {token_ids}")
print(f"Token embedding[3]: {[round(v, 3) for v in tok_vecs[0]]}")
print(f"Pos   embedding[0]: {[round(v, 3) for v in pos_vecs[0]]}")
print(f"Sum (input to GPT): {[round(v, 3) for v in x[0]]}")

# ─────────────────────────────────────────────────────────────
# SECTION 2: GPT Block (Decoder Block)
#
# A GPT block is identical to an Encoder Block with ONE difference:
# the Multi-Head Attention uses a CAUSAL MASK so each token can
# only attend to itself and previous tokens — not future ones.
#
# This is what makes GPT an autoregressive model: it predicts
# one token at a time, always conditioning only on what came before.
# ─────────────────────────────────────────────────────────────

def causal_mha_forward(x, W_q, W_k, W_v, W_o, n_heads, mask):
    """
    A single Multi-Head Causal Self-Attention forward pass
    written as pure functions (no class) for maximum transparency.
    
    x:      [seq_len, d_model]  — input token embeddings
    W_q/k/v: Linear layers for projecting Q, K, V
    W_o:    Output projection Linear layer
    n_heads: Number of attention heads
    mask:   [seq_len, seq_len] lower-triangular causal mask
    """
    seq_len = len(x)
    d_model = len(x[0])
    d_k     = d_model // n_heads

    # Project to Q, K, V (all same shape: [seq_len, d_model])
    Q = W_q(x)
    K = W_k(x)
    V = W_v(x)

    # Split into heads: [n_heads][seq_len][d_k]
    def split(M):
        return [
            [token[h*d_k:(h+1)*d_k] for token in M]
            for h in range(n_heads)
        ]

    Q_h, K_h, V_h = split(Q), split(K), split(V)

    # Per-head attention with causal mask
    head_outs = []
    for h in range(n_heads):
        K_T    = transpose(K_h[h])
        scores = mat_mul(Q_h[h], K_T)
        scale  = math.sqrt(d_k)
        scores = [[s/scale for s in row] for row in scores]

        # Apply causal mask
        for i in range(seq_len):
            for j in range(seq_len):
                if mask[i][j] == 0:
                    scores[i][j] = float('-inf')

        weights = [softmax(row) for row in scores]

        head_out = make_zeros(seq_len, d_k)
        for i in range(seq_len):
            for j in range(seq_len):
                for d in range(d_k):
                    head_out[i][d] += weights[i][j] * V_h[h][j][d]

        head_outs.append(head_out)

    # Concatenate heads: [seq_len, d_model]
    concat = [
        [v for h in range(n_heads) for v in head_outs[h][t]]
        for t in range(seq_len)
    ]

    return W_o(concat)


class GPTBlock:
    """
    One GPT Transformer Block (Decoder block with causal masking).
    
    Uses Pre-LayerNorm (norm applied BEFORE each sublayer, not after).
    """
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads

        self.W_q   = Linear(d_model, d_model)
        self.W_k   = Linear(d_model, d_model)
        self.W_v   = Linear(d_model, d_model)
        self.W_o   = Linear(d_model, d_model)

        self.ff    = Linear(d_model, 4*d_model)   # Expand
        self.ff2   = Linear(4*d_model, d_model)   # Contract

        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

    def __call__(self, x):
        seq_len = len(x)

        # Build causal mask for this sequence length
        mask = make_causal_mask(seq_len)

        # Sub-layer 1: Causal Self-Attention with residual
        normed1  = self.norm1(x)
        attn_out = causal_mha_forward(
            normed1, self.W_q, self.W_k, self.W_v, self.W_o, self.n_heads, mask
        )
        x = [add_vectors(x[t], attn_out[t]) for t in range(seq_len)]

        # Sub-layer 2: Feed-Forward with residual
        normed2 = self.norm2(x)

        # Expand → GELU → Contract
        hidden  = self.ff(normed2)                                   # [seq_len, 4*d_model]
        hidden  = [[gelu(v) for v in token] for token in hidden]    # Non-linearity
        ff_out  = self.ff2(hidden)                                   # [seq_len, d_model]

        x = [add_vectors(x[t], ff_out[t]) for t in range(seq_len)]
        return x


# ─────────────────────────────────────────────────────────────
# SECTION 3: The Full MiniGPT
# ─────────────────────────────────────────────────────────────

def cross_entropy_loss(logits, target_id):
    """
    Cross-entropy loss for a single position.
    
    logits:    [vocab_size] list of raw scores
    target_id: int — the index of the correct next token
    
    Loss = -log(softmax(logits)[target_id])
    
    Intuitively: the loss is low when the model assigns high probability
    to the correct next token, and high when it assigns low probability.
    """
    probs = softmax(logits)
    # Clip to avoid log(0)
    p_correct = max(probs[target_id], 1e-9)
    return -math.log(p_correct)


class MiniGPT:
    """
    A complete character-level GPT model — pure Python.
    
    vocab_size:  Number of unique characters/tokens
    d_model:     Embedding dimension
    n_heads:     Attention heads per block
    n_layers:    Number of stacked GPT blocks
    max_seq_len: Maximum context window
    """
    def __init__(self, vocab_size, d_model=16, n_heads=2, n_layers=2, max_seq_len=32):
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len

        # Embeddings
        self.token_emb = Embedding(vocab_size, d_model)
        self.pos_emb   = Embedding(max_seq_len, d_model)

        # Stack of GPT blocks
        self.blocks = [GPTBlock(d_model, n_heads) for _ in range(n_layers)]

        # Final normalization
        self.final_norm = LayerNorm(d_model)

        # Language model head: project d_model → vocab_size scores
        # NOTE: In real GPT this shares weights with token_emb (weight tying).
        # Here we keep it separate for clarity.
        self.lm_head = Linear(d_model, vocab_size)

    def forward(self, token_ids):
        """
        token_ids: list of integer token IDs, e.g. [3, 7, 2, 5]
        
        Returns:
            logits: [seq_len, vocab_size]  — one score vector per position
            
        Each logits[t] predicts the NEXT token after position t.
        logits[0] predicts what token 1 will be (given only token 0).
        logits[-1] predicts what comes after the last token in the sequence.
        """
        seq_len = len(token_ids)
        assert seq_len <= self.max_seq_len, "Sequence exceeds max_seq_len!"

        # Step 1: Embed tokens
        tok_vecs = self.token_emb(token_ids)
        pos_vecs = self.pos_emb(list(range(seq_len)))

        # Add token + positional embeddings
        x = [add_vectors(tok_vecs[t], pos_vecs[t]) for t in range(seq_len)]

        # Step 2: Pass through each GPT block
        for block in self.blocks:
            x = block(x)

        # Step 3: Final LayerNorm
        x = self.final_norm(x)

        # Step 4: Project to vocabulary size
        logits = self.lm_head(x)  # [seq_len, vocab_size]

        return logits

    def compute_loss(self, token_ids):
        """
        Compute average cross-entropy loss on a sequence.
        
        The training objective: given token_ids[0..t], predict token_ids[t+1].
        - Input:  token_ids[0:-1]  (all but last)
        - Target: token_ids[1:]    (all but first — the NEXT token)
        
        This is why it's called "next-token prediction."
        """
        # We only use all-but-last as input
        logits = self.forward(token_ids[:-1])  # [seq_len-1, vocab_size]
        targets = token_ids[1:]                 # [seq_len-1]

        # Average cross-entropy across all positions
        total_loss = sum(
            cross_entropy_loss(logits[t], targets[t])
            for t in range(len(targets))
        )
        return total_loss / len(targets)

    def generate(self, prompt_ids, n_new_tokens, temperature=1.0):
        """
        Autoregressively generate new tokens given a starting prompt.
        
        At each step:
          1. Run the model on the current context
          2. Get the logits for the LAST position (predicts the next token)
          3. Apply temperature scaling (lower temp = more confident)
          4. Sample from the probability distribution
          5. Append the new token and repeat
        
        This is the EXACT same loop that GPT-4 uses — just scaled to 100B params!
        """
        ids = list(prompt_ids)  # Make a copy

        for step in range(n_new_tokens):
            # Crop to max context window
            context = ids[-self.max_seq_len:]

            # Forward pass — we only use the LAST position's logits
            logits    = self.forward(context)
            last_logit = logits[-1]  # [vocab_size]

            # Temperature scaling: divide logits before softmax
            # Low temp (0.1): very peaked probs → predictable output
            # High temp (2.0): flat probs → random/creative output
            scaled = [l / temperature for l in last_logit]
            probs  = softmax(scaled)

            # Sample next token: weighted random choice
            # Not argmax! Argmax would always pick the same token → boring loops
            r = random.random()
            cumulative = 0.0
            next_id = len(probs) - 1  # Fallback to last token
            for token_id, p in enumerate(probs):
                cumulative += p
                if r < cumulative:
                    next_id = token_id
                    break

            ids.append(next_id)

        return ids  # Full sequence: prompt + newly generated tokens


# ─────────────────────────────────────────────────────────────
# SECTION 4: Full Demo on a Real Mini-Corpus
# ─────────────────────────────────────────────────────────────

# Build a character-level vocabulary from a small text
text = "hello world. the cat sat on the mat. to be or not to be."
chars     = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print("\n\n" + "=" * 55)
print("DEMO: MiniGPT Forward Pass and Text Generation")
print("=" * 55)
print(f"Text:       '{text}'")
print(f"Vocab size: {vocab_size} unique characters")
print(f"Characters: {chars}")

# Initialize a tiny GPT (this is NOT trained — random weights!)
random.seed(42)
model = MiniGPT(
    vocab_size  = vocab_size,
    d_model     = 8,    # Very small: each embedding is 8 numbers
    n_heads     = 2,    # 2 attention heads of 4 dims each
    n_layers    = 2,    # 2 stacked GPT blocks
    max_seq_len = 32
)

# ── Forward pass ──────────────────────────────────────────────
sample = "hello"
ids    = encode(sample)
print(f"\nForward pass on '{sample}':")
print(f"  Token IDs: {ids}")

logits = model.forward(ids)
print(f"  Logits shape: {len(logits)} positions × {len(logits[0])} vocab scores")

# Convert last-position logits to probabilities
probs = softmax(logits[-1])
print(f"\n  After '{sample}', top-3 predicted next chars:")
ranked = sorted(enumerate(probs), key=lambda x: -x[1])[:3]
for char_id, prob in ranked:
    print(f"    '{itos[char_id]}' → {prob:.1%}")
print("  (These are random guesses — the model is not trained!)")

# ── Loss computation ──────────────────────────────────────────
loss = model.compute_loss(ids)
uniform_loss = math.log(vocab_size)  # What random guessing looks like
print(f"\n  Training loss on '{sample}': {loss:.4f}")
print(f"  Uniform random baseline:     {uniform_loss:.4f}")
print(f"  (Loss should drop below {uniform_loss:.2f} as the model trains)")

# ── Text Generation ───────────────────────────────────────────
print(f"\n--- Text Generation (untrained model — expect gibberish) ---")
prompt    = "he"
prompt_ids = encode(prompt)

for temp in [0.5, 1.0, 1.5]:
    gen_ids  = model.generate(prompt_ids, n_new_tokens=20, temperature=temp)
    gen_text = decode(gen_ids)
    print(f"  temp={temp}: '{gen_text}'")

print("\n  After training, the model would generate coherent text!")
print("  The only difference between this and GPT-4 is:")
print("  → GPT-4 uses d_model=12288, n_heads=96, n_layers=96")
print("  → GPT-4 was trained on trillions of tokens for months on 1000+ GPUs")
print("  → The ARCHITECTURE is identical to what you just ran!")

# ─────────────────────────────────────────────────────────────
# SECTION 5: Illustrating the Training Step (Conceptually)
#
# Real training uses backpropagation to compute gradients.
# Writing full backprop in pure Python is complex, but we can
# illustrate the CONCEPT with finite-difference gradient estimation:
#
#   ∂L/∂w ≈ (L(w + ε) - L(w - ε)) / (2ε)
#
# This tells us: "if I nudge weight w by a tiny amount +ε,
# how much does the loss change?"
# In real training, automatic differentiation does this analytically
# for every weight simultaneously — billions of parameters at once.
# ─────────────────────────────────────────────────────────────

print("\n\n" + "=" * 55)
print("DEMO: Finite-Difference Gradient (1 weight, illustrative)")
print("=" * 55)

training_text = "hello"
training_ids  = encode(training_text)

loss_before = model.compute_loss(training_ids)
print(f"Loss before:  {loss_before:.6f}")

# Pick one specific weight: the first weight in lm_head's W matrix
original_value = model.lm_head.W[0][0]
epsilon        = 1e-4

# Nudge up
model.lm_head.W[0][0] = original_value + epsilon
loss_up = model.compute_loss(training_ids)

# Nudge down
model.lm_head.W[0][0] = original_value - epsilon
loss_down = model.compute_loss(training_ids)

# Finite-difference gradient estimate
gradient = (loss_up - loss_down) / (2 * epsilon)
print(f"Finite-difference gradient of lm_head.W[0][0]: {gradient:.4f}")

# Gradient descent step: move the weight AGAINST the gradient
learning_rate = 0.01
new_value     = original_value - learning_rate * gradient
model.lm_head.W[0][0] = new_value

loss_after = model.compute_loss(training_ids)
print(f"Loss after one gradient step: {loss_after:.6f}")

if loss_after < loss_before:
    print("✓ Loss went DOWN — gradient step worked!")
else:
    print("→ Loss didn't change much — one weight out of many doesn't matter much!")
print("\n  Real training does this for ALL weights simultaneously using backprop.")